In [ ]:
!pip install sklearn pandas numpy matplotlib seaborn webdriver-manager beautifulsoup4 tensorflow

Modelo Random Forest

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv("tabela_multi_hot.csv")   
df.head(20)

df2= df.drop(columns=['Farmaco', 'Subgrupo', 'Forma_farmaceutica','Indicacao'])
df2.head(20)

In [ ]:
le = LabelEncoder()
df2["doenca_encoded"] = le.fit_transform(df2["diagnostico_regra"])


X = df2.drop(columns=["diagnostico_regra", "doenca_encoded"])
y = df2["doenca_encoded"]
X.head(20), y.head(20)


In [ ]:
df2["doenca_encoded"].value_counts()


In [ ]:
# remover as doencas que só tem uma linha pq nao da pra dividir em treino e teste

count_classes = y.value_counts()

classes_ok = count_classes[count_classes >= 2].index

mask = y.isin(classes_ok)
X_filtrado = X[mask]
y_filtrado = y[mask]

print("Formato antes:", X.shape, "Depois do filtro:", X_filtrado.shape)
print("Quantidade por classe depois do filtro:")
print(y_filtrado.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_filtrado,
    y_filtrado,
    test_size=0.25,
    random_state=42,
    stratify=y_filtrado
)


In [ ]:
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    class_weight="balanced" 
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)


In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Acurácia no conjunto de teste: {acc:.4f}\n")

print("CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))


In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de Confusão — Random Forest")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.show()


In [ ]:
importancias = pd.DataFrame({
    "Sintoma": X_train.columns,
    "Importancia": rf.feature_importances_
}).sort_values(by="Importancia", ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=importancias.head(20), x="Importancia", y="Sintoma")
plt.title("Top 20 sintomas mais importantes")
plt.show()

importancias.head(20)


: 